In [50]:
import pandas as pd
import altair as alt

In [51]:
df = pd.read_csv('ds4200_data.csv')
df.head()

,Timestamp,How old are you?,How would you best describe yourself?,Are you currently a student?,What city do you live in?,"For the following list of movies, please select if/where you first watched each movie. [Wicked]","For the following list of movies, please select if/where you first watched each movie. [Zootopia 2]","For the following list of movies, please select if/where you first watched each movie. [Sinners]","For the following list of movies, please select if/where you first watched each movie. [Oppenheimer]","For the following list of movies, please select if/where you first watched each movie. [Superman]","For the following list of movies, please select if/where you first watched each movie. [No Other Choice]","For the following list of movies, please select if/where you first watched each movie. [One Battle After Another]","For the following list of movies, please select if/where you first watched each movie. [Dune 2]","For the following list of movies, please select if/where you first watched each movie. [Marty Supreme]","For the following list of movies, please select if/where you first watched each movie. [Barbie]",How many movies have you seen in theaters in the past 3 months (estimate)?,How many movies have you watched on streaming in the past 3 months (estimate)?,Which streaming services are you subscribed to? (Select all that apply),What is your biggest barrier to going to the movie theater?
0,2/11/26 15:34,20,Female,Yes,"Boston, Massachusetts",Watched in theaters,Watched in theaters,Watched on streaming,Watched on streaming,Watched in theaters,Did not watch,Did not watch,Did not watch,Watched in theaters,Watched in theaters,6 to 10,1 to 5,"Netflix, Hulu, HBO Max, Disney+, AppleTV",No time
1,2/18/26 09:26,21,Female,Yes,"Boston, Massachusetts",Did not watch,Did not watch,Did not watch,Did not watch,Watched on streaming,Did not watch,Did not watch,Did not watch,Did not watch,Watched in theaters,0,1 to 5,Netflix,Cost of tickets
2,2/18/26 09:29,21,Male,Yes,"Boston, Massachusetts",Watched in theaters,Did not watch,Watched in theaters,Watched in theaters,Watched in theaters,Did not watch,Watched in theaters,Watched in theaters,Watched in theaters,Watched in theaters,more than 10,6 to 10,"Netflix, Hulu, HBO Max, Disney+, Peacock",Nothing - I always go to the theater!
3,2/18/26 09:30,21,Female,Yes,"Boston, Massachusetts",Watched in theaters,Watched in theaters,Watched in theaters,Watched on streaming,Watched in theaters,Did not watch,Did not watch,Watched in theaters,Watched in theaters,Watched in theaters,6 to 10,1 to 5,"Netflix, Hulu",Cost of tickets
4,2/18/26 09:30,18,Male,Yes,"Middletown, CT",Watched on streaming,Did not watch,Watched in theaters,Watched in theaters,Watched on streaming,Did not watch,Did not watch,Did not watch,Watched in theaters,Watched in theaters,1 to 5,0,"Netflix, Hulu, HBO Max, Disney+, AppleTV, Peacock",Theater is far away


In [52]:
movie_df = df.iloc[:, 5:15]
movie_df.columns = [x[x.index('[')+1:x.index(']')] for x in movie_df.columns]
movie_data = (movie_df == "Watched in theaters").mean().to_frame()
movie_data = movie_data.rename(columns={0:'% Watched in Theaters'})
movie_data.index.name = 'Movie'
movie_data

,% Watched in Theaters
Movie,
Wicked,0.512397
Zootopia 2,0.214876
Sinners,0.247934
Oppenheimer,0.396694
Superman,0.347107
No Other Choice,0.016529
One Battle After Another,0.074380
Dune 2,0.272727
Marty Supreme,0.289256


In [53]:
extra_movie_data = pd.read_csv('movies_data.csv')
extra_movie_data.set_index('Movie', inplace=True)
extra_movie_data
merged = movie_data.merge(extra_movie_data, left_index=True, right_index=True)
merged.reset_index(inplace=True)
merged

,Movie,% Watched in Theaters,Budget (Millions USD),RT Critics Score (%),RT Audience Score (%)
0,Wicked,0.512397,145,88,95
1,Zootopia 2,0.214876,150,93,96
2,Sinners,0.247934,95,97,97
3,Oppenheimer,0.396694,100,93,91
4,Superman,0.347107,225,82,96
5,No Other Choice,0.016529,12,97,93
6,One Battle After Another,0.074380,150,94,88
7,Marty Supreme,0.289256,65,93,83
8,Barbie,0.652893,145,88,83


In [58]:
radio = alt.binding_radio(options = ['Budget (Millions USD)', 'RT Critics Score (%)', 'RT Audience Score (%)'],
                                   labels = ['Budget (Millions USD)', 'RT Critic Score', 'RT Audience Score'],
                                   name = 'x Axis: ')

selection = alt.param(name = 'x_col', value='Budget (Millions USD)', bind = radio)
chart = alt.Chart(merged).mark_point(size=100).add_params(selection).transform_calculate(
    x_choice = 'datum[x_col]').encode(
    y = '% Watched in Theaters:Q',
    tooltip = ['Movie:N', '% Watched in Theaters:Q', 'x_choice:Q'],
    x=alt.X('x_choice:Q', title='')
).properties(title='Theater attendance of recent movies')
chart

alt.Chart(...)

In [59]:
chart.save('vis2_movie_scatter_plot.html')